# 02 - Panel Descriptives

Explore the merged district-year panel before running any causal models.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

## Raw correlation: PM2.5 and test scores (biased baseline)

In [ ]:
corr = panel[["pm25_annual_mean","test_score_mean"]].corr().iloc[0,1]
print(f"Raw corr(PM2.5, test score) = {corr:.3f}")
print("Positive or near-zero = confounded (poor districts have high PM2.5 AND low scores)")

fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(panel["pm25_annual_mean"], panel["test_score_mean"],
           alpha=0.1, s=5, c="steelblue")
ax.set_xlabel("Annual Mean PM2.5 (μg/m³)")
ax.set_ylabel("Test Score (standardized)")
ax.set_title(f"Raw Association: PM2.5 vs Test Scores\ncorr = {corr:.3f}  (confounded by district poverty)")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_raw_scatter.png", bbox_inches="tight")
plt.show()

## PM2.5 trends over time

In [ ]:
annual_pm = panel.groupby("year")["pm25_annual_mean"].mean().reset_index()
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(annual_pm["year"], annual_pm["pm25_annual_mean"], marker="o", lw=1.8)
ax.set_xlabel("Year")
ax.set_ylabel("Mean PM2.5 (μg/m³)")
ax.set_title("Western US District PM2.5 Over Time\n(includes wildfire contribution)")
plt.tight_layout()
plt.savefig(OUT_DIR / "02_pm25_trend.png", bbox_inches="tight")
plt.show()

## DAG: why OLS fails

In [ ]:
print("""
District poverty ──→ PM2.5 exposure ──→ Test scores
       │                                     ↑
       └─────────────────────────────────────┘
              (direct path through resources)

Poor districts: high industrial PM2.5, worse schools, lower scores
Wildfire smoke: relatively democratic — affects rich and poor districts alike
                AND is driven by upwind fire conditions, not local economy

OLS estimate of PM2.5 -> test scores is biased upward (toward zero or positive)
because poverty confounds both sides.

IV fix: use smoke_days as instrument for PM2.5
  Relevance: smoke -> PM2.5 (first stage)
  Exclusion: smoke affects scores only through air quality (conditional on controls)
""")